mlflow ui --backend-store-uri sqlite:////workspaces/Mlops-Zoomcamp/mlflow.db

In [2]:
!python -V

Python 3.9.25


In [13]:
import pandas as pd

In [14]:
import pickle

In [15]:
import seaborn as sns
import matplotlib.pyplot as plt

Intento hacer todo de nuevo

Todas las carpetas de MLFLOW deberian esar en la raiz del proyecto, no en una carpeta carpeta separada

In [16]:
import mlflow
from pathlib import Path

# Sube un nivel a la raíz del repositorio para leer/crear la DB
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

experiment_name = "riesgo_crediticio_xgboost_v1"

# Sube un nivel para guardar los artefactos en la raíz
artifact_path = Path("../mlruns").resolve().as_uri() 

if not mlflow.get_experiment_by_name(experiment_name):
    mlflow.create_experiment(
        name=experiment_name,
        artifact_location=artifact_path
    )

mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='file:///workspaces/Mlops-Zoomcamp/mlruns', creation_time=1779374381515, experiment_id='1', last_update_time=1779374381515, lifecycle_stage='active', name='riesgo_crediticio_xgboost_v1', tags={}>

Luego ejecuto esto en la terminal desde la raiz del proyecto
cd /workspaces/Mlops-Zoomcamp/
mlflow ui --backend-store-uri sqlite:///mlflow.db

Ahora intentamos correr algunos experimentos

In [17]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import root_mean_squared_error

In [18]:
df = pd.read_parquet('../data/green_tripdata_2021-01.parquet')

df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

df = df[(df.duration >= 1) & (df.duration <= 60)]

categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

df[categorical] = df[categorical].astype(str)

In [19]:
train_dicts = df[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

target = 'duration'
y_train = df[target].values

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)

root_mean_squared_error(y_train, y_pred)

9.838799799829577

In [20]:
def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)

        df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
        df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    elif filename.endswith('.parquet'):
        df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [21]:
df_train = read_dataframe('../data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('../data/green_tripdata_2021-02.parquet')

len(df_train), len(df_val)

(73908, 61921)

In [22]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values


lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

7.758715209663881

In [23]:
with open('../models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

# Part 2 Using MlFLow


In [24]:
import xgboost as xgb

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

import gc
import xgboost as xgb
import mlflow
from sklearn.model_selection import train_test_split


In [25]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [26]:
def objective(params):

    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)

        booster = xgb.train(
            params = params,
            dtrain = train,
            num_boost_round=100,
            evals = [(valid, "validation")],
            early_stopping_rounds = 50
        )

        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)

        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [ ]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0), # (exp(-3), exp(0))
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42,
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

/home/codespace/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [14:59:08] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:9.26626                           
[1]	validation-rmse:7.87020                           
[2]	validation-rmse:7.25428                           
[3]	validation-rmse:6.98131                           
[4]	validation-rmse:6.85825                           
[5]	validation-rmse:6.79363                           
[6]	validation-rmse:6.75462                           
[7]	validation-rmse:6.73205                           
[8]	validation-rmse:6.70965                           
[9]	validation-rmse:6.69787                           
[10]	validation-rmse:6.69082                          
[11]	validation-rmse:6.68647                          
[12]	validation-rmse:6.68384                          
[13]	validation-rmse:6.68291                          
[14]	validation-rmse:6.68186                          
[15]	validation-rmse:6.68037                          
[16]	validation-rmse:6.67919                          
[17]	validation-rmse:6.67766                          
[18]	valid

/home/codespace/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [14:59:36] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:8.84365                                                   
[1]	validation-rmse:7.42698                                                   
[2]	validation-rmse:6.88146                                                   
[3]	validation-rmse:6.66307                                                   
[4]	validation-rmse:6.56339                                                   
[5]	validation-rmse:6.51406                                                   
[6]	validation-rmse:6.48710                                                   
[7]	validation-rmse:6.47363                                                   
[8]	validation-rmse:6.46711                                                   
[9]	validation-rmse:6.45869                                                   
[10]	validation-rmse:6.45405                                                  
[11]	validation-rmse:6.44845                                                  
[12]	validation-rmse:6.44598                        

/home/codespace/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [15:00:11] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



: 

In [36]:
with mlflow.start_run():
    alpha = 0.01
    mlflow.set_tag("model", "LinearRegression")
    mlflow.log_param("alpha", alpha)
    mlflow.log_param("train_data", "../data/green_tripdata_2021-01.parquet")
    mlflow.log_param("val_data", "../data/green_tripdata_2021-02.parquet")
    
    alpha = 0.01
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)


    rmse =root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(lr, artifact_path="model_pickle")

2026/05/21 14:56:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/21 14:57:02 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Suponemos que el siguiente es el mejor

In [ ]:
params= {
    'learning_rate': 0.44855137273086154,
    'max_depth': 56,
    'min_child_weight': 1.1760648012996973,
    'objective': 'reg:linear',
    'reg_alpha': 0.2017903944755146,
    'reg_lambda': 0.12094781555192394,
    'seed': 42
}

with mlflow.start_run():

    mlflow.set_tag("model", "xgboost")
    mlflow.log_params(params)

    booster = xgb.train(
        params = params,
        dtrain = train,
        num_boost_round=100,
        evals = [(valid, "validation")],
        early_stopping_rounds = 50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)

    mlflow.log_metric("rmse", rmse)

Tambien se puede usar autolog

In [17]:
params= {
    'learning_rate': 0.44855137273086154,
    'max_depth': 56,
    'min_child_weight': 1.1760648012996973,
    'objective': 'reg:linear',
    'reg_alpha': 0.2017903944755146,
    'reg_lambda': 0.12094781555192394,
    'seed': 42
}

mlflow.xgboost.autolog()

booster = xgb.train(
    params = params,
    dtrain = train,
    num_boost_round=100,
    evals = [(valid, "validation")],
    early_stopping_rounds = 50
)

2026/05/21 16:18:29 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'ee3c237751b04c6ba4c12cb034870f9b', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current xgboost workflow


: 

Por alguna razon el autolog te genera una roptura de kernel en este sistema de github. Pero en la vida real se podria usar

# Como guardar un modelo

Asi se guarda un modelo de entrada usando pickle, y se guarda en arfiacts

In [29]:
with mlflow.start_run():

    mlflow.set_tag("model", "LR")
    mlflow.log_param("train-data-path", "../data/green_tripdata_2021-01.parquet")
    mlflow.log_param("val-data-path", "../data/green_tripdata_2021-02.parquet")

    alpha = 0.01

    mlflow.log_params({"alpha": alpha})

    lr = Lasso(alpha)
    lr.fit(X_train, y_train)


    y_pred = lr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)

    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact('../models/lin_reg.bin', artifact_path="model_pickle")


Otra forma de guardar un modelo de una mejor forma

In [30]:

params = {
    'learning_rate': 0.44855137273086154,
    'max_depth': 56,
    'min_child_weight': 1.1760648012996973,
    'objective': 'reg:linear',
    'reg_alpha': 0.2017903944755146,
    'reg_lambda': 0.12094781555192394,
    'seed': 42
}

with mlflow.start_run():
    mlflow.log_params(params)

    booster = xgb.train(
        params = params,
        dtrain = train,
        num_boost_round=100,
        evals = [(valid, "validation")],
        early_stopping_rounds = 50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    #Y ahora guardamos el modelo


IndentationError: unexpected indent (508319005.py, line 2)